## Production Notebook
Create a separate notebook that has a function similar to below.

You will receive "prod" data before the class starts.

In [1]:
import pandas as pd
# don't use this
from random import randint

class TestModel:
    def predict(self, X):
        return [ randint(0, 1) for _ in range(len(X))]

In [5]:
# In the second notebook

from sklearn.metrics import classification_report

def production(X_path, y_path):
    # load model
    from joblib import load
    model = load("best_xgb_pipeline.pkl")

    
   # load production data  
    df_X = pd.read_csv(X_path)  
    df_y = pd.read_csv(y_path)['Left']  

    print("Loaded data with shape:", df_X.shape)  
    print("Available columns:", df_X.columns.tolist())  

    # -- Feature Engineering (same as training) --  
    df_X['PreviousSalary'] = pd.to_numeric(df_X['PreviousSalary'].str.replace('K', ''), errors='coerce') * 1000  
    df_X['Salary'] = pd.to_numeric(df_X['Salary'].str.replace('K', ''), errors='coerce') * 1000  

    df_X['Sal_Increase'] = df_X['Salary'] - df_X['PreviousSalary']  
    df_X['Sal_ratio'] = df_X['Salary'] / df_X['PreviousSalary']  
    df_X['Sal_ch_perc'] = ((df_X['Salary'] - df_X['PreviousSalary']) / df_X['PreviousSalary']) * 100  

    df_X['Distance'] = df_X['Distance'].replace({  
        '<5mile': '<5mile',  
        '~10miles': '~10miles',  
        '~15miles': '~15miles',  
        '~20miles': '~20miles',  
        '>30miles': '>30miles'  
    })  

    df_X['diffrev'] = df_X['SelfReview'] - df_X['SupervisorReview']  
    df_X['Effortlevel'] = df_X['TrainingHours'] / (df_X['YearsWorked'] + 1)  

    sat_cols = ['WorkSatisfactionScore', 'JobEngagementScore', 'MentalWellbeingScore', 'PhysicalActivityScore']  
    df_X['aggsat'] = df_X[sat_cols].mean(axis=1)  
    df_X['Resilev'] = df_X['aggsat'] - df_X['StressLevel']  
    df_X['loadIndex'] = (df_X['NumOfProjects'] * df_X['ProjectComplexity']) / (df_X['TeamSize'] + 1)  
    df_X['Wlindex'] = df_X['loadIndex']  
    df_X['Wlindica'] = df_X['WorkLifeBalance'] - df_X['Wlindex']  

    # Keep only the features used during training  
    selected_features = [  
        'Sal_Increase', 'Sal_ratio', 'Sal_ch_perc', 'Effortlevel',  
        'diffrev', 'aggsat', 'Resilev', 'loadIndex', 'Wlindex', 'Wlindica',  
        'YearsWorked', 'TrainingHours', 'SelfReview', 'SupervisorReview',  
        'AttendanceRate', 'Distance', 'Gender', 'WorkLifeBalance'  
    ]  
    df_X = df_X[selected_features]  

    # -- Predict and evaluate --  
    y_pred = model.predict(df_X)  

    print("\n Classification Report on Production Data:")  
    print(classification_report(df_y, y_pred))  


production( 

  X_path='https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/employee_departure_dataset_X_prod.csv',
  y_path='https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/employee_departure_dataset_y_prod.csv'

)

Loaded data with shape: (100000, 27)
Available columns: ['RecordId', 'Gender', 'Distance', 'YearsWorked', 'TrainingHours', 'WorkLifeBalance', 'NumOfProjects', 'JobInvolvement', 'TeamSize', 'MentorshipReceived', 'TechSkillLevel', 'AttendanceRate', 'StressLevel', 'PeerFeedbackScore', 'AnnualLeaveDays', 'Certifications', 'SkillDevelopmentCourses', 'ProjectComplexity', 'WorkSatisfactionScore', 'JobEngagementScore', 'PhysicalActivityScore', 'MentalWellbeingScore', 'DepartmentCode', 'PreviousSalary', 'Salary', 'SelfReview', 'SupervisorReview']

 Classification Report on Production Data:
              precision    recall  f1-score   support

           0       0.84      0.73      0.78     64044
           1       0.61      0.75      0.68     35956

    accuracy                           0.74    100000
   macro avg       0.73      0.74      0.73    100000
weighted avg       0.76      0.74      0.74    100000

